In [172]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords,wordnet
from nltk.stem import PorterStemmer,WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle
import re

In [173]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HARIHARAN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HARIHARAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HARIHARAN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HARIHARAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [174]:
df = pd.read_csv("megaGymDataset.csv")
df.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [175]:
df.describe()

,Unnamed: 0,Rating
count,2918.000000,1031.000000
mean,1458.500000,5.919690
std,842.498368,3.584607
min,0.000000,0.000000
25%,729.250000,3.000000
50%,1458.500000,7.900000
75%,2187.750000,8.700000
max,2917.000000,9.600000


In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2918 non-null   int64  
 1   Title       2918 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        2918 non-null   object 
 4   BodyPart    2918 non-null   object 
 5   Equipment   2886 non-null   object 
 6   Level       2918 non-null   object 
 7   Rating      1031 non-null   float64
 8   RatingDesc  862 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 205.3+ KB


In [177]:
df.isnull().sum()

Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64

In [178]:
df = df.dropna(subset = ["Desc"])

In [179]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1368 entries, 0 to 2916
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  1368 non-null   int64  
 1   Title       1368 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        1368 non-null   object 
 4   BodyPart    1368 non-null   object 
 5   Equipment   1359 non-null   object 
 6   Level       1368 non-null   object 
 7   Rating      595 non-null    float64
 8   RatingDesc  506 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 106.9+ KB


In [180]:
df["RatingDesc"].value_counts()

RatingDesc
Average    506
Name: count, dtype: int64

In [181]:
df = df.drop(columns = ["Unnamed: 0","Rating","RatingDesc"],axis = 1)

In [182]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1368 entries, 0 to 2916
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Title      1368 non-null   object
 1   Desc       1368 non-null   object
 2   Type       1368 non-null   object
 3   BodyPart   1368 non-null   object
 4   Equipment  1359 non-null   object
 5   Level      1368 non-null   object
dtypes: object(6)
memory usage: 74.8+ KB


In [183]:
df.isnull().sum()

Title        0
Desc         0
Type         0
BodyPart     0
Equipment    9
Level        0
dtype: int64

In [184]:
df[df["Equipment"].isna()]

,Title,Desc,Type,BodyPart,Equipment,Level
637,Decline oblique crunch,The decline oblique crunch is a popular bodywe...,Strength,Abdominals,NaN,Intermediate
638,Decline sit-up,The decline sit-up is a bodyweight core exerci...,Strength,Abdominals,NaN,Intermediate
639,Hanging Windshield Wiper,The hanging windshield wiper is an advanced ab...,Strength,Abdominals,NaN,Intermediate
1402,Glute ham raise-,The glute ham raise is an exercise targeting t...,Strength,Hamstrings,NaN,Beginner
1403,Lying hamstring stretch with band,The lying hamstring stretch with band is a sim...,Stretching,Hamstrings,NaN,Beginner
1406,Alternating lunge jump,The alternating lunge jump is an explosive bod...,Stretching,Hamstrings,NaN,Beginner
2421,Dumbbell lateral hop to sprint,The dumbbell lateral hop to sprint is a multi-...,Plyometrics,Quadriceps,NaN,Intermediate
2422,Smith machine lunge sprint,The Smith machine lunge sprint is a lower-body...,Strength,Quadriceps,NaN,Intermediate
2423,Sissy squat,The sissy squat is a bodyweight squat variatio...,Strength,Quadriceps,NaN,Intermediate


In [185]:
df.Equipment.unique()

array(['Bands', 'Barbell', 'Kettlebells', 'Dumbbell', 'Other', 'Cable',
       'Machine', 'Body Only', 'Medicine Ball', nan, 'Exercise Ball',
       'Foam Roll', 'E-Z Curl Bar'], dtype=object)

In [186]:
df.loc[637, "Equipment"] = "Body Only"
df.loc[638, "Equipment"] = "Body Only"
df.loc[639, "Equipment"] = "Body Only"
df.loc[1402, "Equipment"] = "Body Only"
df.loc[1403, "Equipment"] = "Bands"
df.loc[1406, "Equipment"] = "Body Only"
df.loc[2421, "Equipment"] = "Dumbbell"
df.loc[2422, "Equipment"] = "Other"
df.loc[2423, "Equipment"] = "Body Only"

In [187]:
df.Equipment.unique()

array(['Bands', 'Barbell', 'Kettlebells', 'Dumbbell', 'Other', 'Cable',
       'Machine', 'Body Only', 'Medicine Ball', 'Exercise Ball',
       'Foam Roll', 'E-Z Curl Bar'], dtype=object)

In [188]:
df.Title.value_counts()

Title
Seated Cable Rows                        3
Band-suspended kettlebell bench press    3
Seated rear delt fly                     2
Dumbbell step-up                         2
Arnold press                             2
                                        ..
AM Flat Bench Barbell Press              1
TBS Close-Grip Bench Press               1
Bench press                              1
Incline bench press                      1
TBS Skullcrusher                         1
Name: count, Length: 1359, dtype: int64

In [189]:
df.Desc.value_counts()

Desc
The barbell back squat is a popular compound movement that emphasizes building the lower-body muscle groups and overall strength. It's the classic way to start a leg day, and is a worthy centerpiece to a lower-body training program. The squat is a competitive lift in the sport of powerlifting, but is also a classic measurement of lower-body strength. With the barbell racked on the traps or upper back, the emphasis is placed on the posterior chain but the entire body gets worked. The back squat can be trained in everything from heavy singles to sets of 20 reps or higher.    10
The barbell stiff-legged deadlift targets the hamstrings, glutes, lower and upper back, as well as the core. It is a popular accessory movement for the deadlift, but also a muscle-building hamstring movement.                                                                                                                                                                                                            

In [190]:
df = df.drop_duplicates(subset = ["Title"])

In [191]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1359 entries, 0 to 2916
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Title      1359 non-null   object
 1   Desc       1359 non-null   object
 2   Type       1359 non-null   object
 3   BodyPart   1359 non-null   object
 4   Equipment  1359 non-null   object
 5   Level      1359 non-null   object
dtypes: object(6)
memory usage: 74.3+ KB


In [192]:
df["Title"] = df["Title"].str.replace(
    r"^30\s+",
    "",
    regex=True
)

In [193]:
text_desc = "The barbell back squat is a popular compound movement that emphasizes building the lower-body muscle groups and overall strength. It's the classic way to start a leg day, and is a worthy centerpiece to a lower-body training program. The squat is a competitive lift in the sport of powerlifting, but is also a classic measurement of lower-body strength. With the barbell racked on the traps or upper back, the emphasis is placed on the posterior chain but the entire body gets worked. The back squat can be trained in everything from heavy singles to sets of 20 reps or higher."

In [194]:
df[df["Desc"]== text_desc]

,Title,Desc,Type,BodyPart,Equipment,Level
1799,Barbell Full Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1810,Barbell Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1822,Barbell back squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1849,Paul Carter Back Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1851,TBS High-Bar Back Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1853,Squat - Gethin Variation,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1857,Barbell Squat - Gethin Variation,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1859,AM Barbell Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1862,UP Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate
1867,King Maker Barbell Back Squat,The barbell back squat is a popular compound m...,Strength,Quadriceps,Barbell,Intermediate


In [195]:
df["Actual_desc"] = (
    df["Desc"].astype(str) + " | " +
    df["Type"].astype(str) + " | " +
    df["BodyPart"].astype(str) + " | " +
    df["Equipment"].astype(str) + " | " +
    df["Level"].astype(str)
)

In [196]:
df.shape

(1359, 7)

In [197]:
df = df.reset_index(drop = True)

In [198]:
df.head()

,Title,Desc,Type,BodyPart,Equipment,Level,Actual_desc
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,The partner plank band row is an abdominal exe...
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,The banded crunch isometric hold is an exercis...
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,The banded plank jack is a variation on the pl...
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,The banded crunch is an exercise targeting the...
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,The crunch is a popular core exercise targetin...


In [199]:
df["Actual_desc"][1]

'The banded crunch isometric hold is an exercise targeting the abdominal muscles, particularly the rectus abdominis or "six-pack" muscles. The band adds resistance and continuous tension to this popular exercise. | Strength | Abdominals | Bands | Intermediate'

In [200]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [201]:
def clean(text):
    tokens = word_tokenize(text.lower())
    processed_tokens = []
    for token in tokens:
        if token.isalpha() and token not in stop_words:
            stemmed_token = stemmer.stem(token)
            lemmatized_token  = lemmatizer.lemmatize(stemmed_token)
            processed_tokens.append(lemmatized_token)

    return " ".join(processed_tokens)
    

In [202]:
df["Actual_desc"] = df["Actual_desc"].apply(clean)

In [203]:
df.head()

,Title,Desc,Type,BodyPart,Equipment,Level,Actual_desc
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,partner plank band row abdomin exercis two par...
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,band crunch isometr hold exercis target abdomi...
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,band plank jack variat plank involv move leg r...
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,band crunch exercis target abdomin muscl parti...
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,crunch popular core exercis target rectu abdom...


In [204]:
df["Actual_desc"][1]

'band crunch isometr hold exercis target abdomin muscl particularli rectu abdomini muscl band add resist continu tension popular exercis strength abdomin band intermedi'

In [205]:
#Vectorizer = CountVectorizer(stop_words = "english")
#vectors = Vectorizer.fit_transform(df["Actual_desc"]).toarray()

In [206]:
Cleaned_df = df[["Title","Actual_desc"]]

In [207]:
Vectorizer = CountVectorizer(stop_words = "english")
vectors = Vectorizer.fit_transform(Cleaned_df["Actual_desc"]).toarray()

In [208]:
vectors

array([[0, 0, 2, ..., 0, 0, 0],
       [0, 0, 2, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [209]:
vectors.shape

(1359, 1280)

In [210]:
similarity = cosine_similarity(vectors)

In [211]:
similarity[0]

array([1.        , 0.53526436, 0.50832857, ..., 0.2245251 , 0.19179882,
       0.19179882])

In [212]:
Vectorizer.get_feature_names_out(10)

array(['ab', 'abdomen', 'abdomin', ..., 'yoga', 'zercher', 'zero'],
      dtype=object)

In [213]:
def pred(title):
    index = Cleaned_df[Cleaned_df["Title"] == title].index[0]
    list_var = list(enumerate(similarity[index]))
    sorted_list = sorted(list_var ,key = lambda x:x[1] , reverse = True)
    top_five = sorted_list[1:6]

    for i in top_five:
        print (Cleaned_df.iloc[i[0]].Title)

In [214]:
Cleaned_df[Cleaned_df["Title"] == "Decline bar press sit-up"].index[0]



9

In [215]:
sorted(list(enumerate(similarity[0])),reverse = True,key = lambda x:x[1])

[(0, 1.0),
 (791, 0.9718253158075502),
 (585, 0.5527707983925667),
 (3, 0.5512459105263765),
 (1, 0.5352643613280605),
 (1271, 0.5266851623825876),
 (2, 0.5083285677753488),
 (6, 0.50709255283711),
 (838, 0.5),
 (1275, 0.47628967220784013),
 (181, 0.4720587952499556),
 (198, 0.4720587952499556),
 (314, 0.4719399037242694),
 (146, 0.46423834544262965),
 (123, 0.4622501635210242),
 (330, 0.4517812304569955),
 (1274, 0.4517812304569955),
 (153, 0.4422689813358516),
 (4, 0.4409585518440984),
 (128, 0.4409585518440984),
 (251, 0.4409585518440983),
 (793, 0.4383972994809528),
 (279, 0.4371928248925937),
 (289, 0.4371928248925937),
 (290, 0.4371928248925937),
 (130, 0.433289122413121),
 (131, 0.433289122413121),
 (176, 0.43259045634870014),
 (255, 0.4320792826090466),
 (272, 0.42426406871192845),
 (164, 0.4170288281141496),
 (124, 0.41666666666666674),
 (586, 0.41328414257736795),
 (682, 0.4132841425773679),
 (214, 0.41147559989891175),
 (860, 0.40857528153788336),
 (749, 0.39999999999999997)

In [216]:
pred("Hammer Curls")


FYR Dumbbell Hammer Curl
Dumbbell Hammer Curl - Gethin Variation
AM Hammer Curls
Alternate Hammer Curl
TBS Hammer Curl


In [217]:
pred("FYR Banded Plank Jack")

Spider plank jack
Partner plank band row
Partner side plank band row
Plank up-down
Plank Push-Up


In [218]:
df["BodyPart"].unique()

array(['Abdominals', 'Abductors', 'Adductors', 'Biceps', 'Calves',
       'Chest', 'Forearms', 'Glutes', 'Hamstrings', 'Lats', 'Lower Back',
       'Middle Back', 'Traps', 'Quadriceps', 'Shoulders', 'Triceps'],
      dtype=object)

In [219]:
df[df["BodyPart"] == "Chest"]

,Title,Desc,Type,BodyPart,Equipment,Level,Actual_desc
438,Band-suspended kettlebell bench press,The band-suspended kettlebell bench press is a...,Strength,Chest,Bands,Intermediate,kettlebel bench press bench press variat kettl...
439,Incline band bench press,The incline band bench press is variation of t...,Strength,Chest,Bands,Intermediate,inclin band bench press variat inclin press po...
440,Band push-up,The band push-up is a progression of the popul...,Strength,Chest,Bands,Intermediate,band progress popular bodyweight version exerc...
441,Band chest fly,"Similar to the cable chest fly, the band chest...",Strength,Chest,Bands,Intermediate,similar cabl chest fli band chest fli movement...
442,Close-grip bench press,The close-grip bench press is a popular exerci...,Strength,Chest,Barbell,Intermediate,bench press popular exercis target tricep ches...
...,...,...,...,...,...,...,...
580,Bar Push-Up Smith Machine,The hands-elevated push-up is a variation on t...,Plyometrics,Chest,Other,Beginner,variat standard hand elev bodi align angl floo...
581,Plate-weighted push-up,The plate-weighted push-up is a simple way to ...,Strength,Chest,Other,Intermediate,simpl way make classic bodyweight exercis diff...
582,Double-bar roll-out chest fly,The double-bar roll-out chest fly is a chest e...,Strength,Chest,Other,Intermediate,chest fli chest exercis util two rotat barbel ...
583,Exercise ball chest stretch,The exercise ball chest stretch is a simple st...,Stretching,Chest,Exercise Ball,Beginner,exercis ball chest stretch simpl stretch pecto...


In [220]:
pred("Bench press")


Barbell Bench Press - Medium Grip
AM Flat Bench Barbell Press
UP Bench Press
King Maker Barbell Bench Press
Close-grip bench press


In [221]:
df[df["Title"] == "Single-arm dumbbell front squat"]

,Title,Desc,Type,BodyPart,Equipment,Level,Actual_desc
947,Single-arm dumbbell front squat,The single-arm dumbbell front squat is an exer...,Strength,Quadriceps,Dumbbell,Intermediate,dumbbel front squat exercis target quad glute ...


In [222]:
df[df["BodyPart"] == "Lats"]

,Title,Desc,Type,BodyPart,Equipment,Level,Actual_desc
680,Band-assisted pull-up,The band-assisted pull-up is a variation of th...,Strength,Lats,Bands,Intermediate,variat exercis rep perform elast band loop aro...
681,Assisted Chin-Up,The reverse-grip chin-up is a variation of the...,Strength,Lats,Bands,Beginner,variat exercis rep perform palm face toward bo...
682,Single-arm band low row,The single-arm band low row is a single-arm ro...,Strength,Lats,Bands,Intermediate,band low row row variat util band resist ancho...
683,Latissimus dorsi SMR,Latissimus dorsi self-myofascial release (SMR)...,Stretching,Lats,Foam Roll,Intermediate,latissimu dorsi releas smr treatment upper bod...
684,Bent-arm barbell pull-over,The bent-arm barbell pull-over was a staple ex...,Strength,Lats,Barbell,Intermediate,barbel stapl exercis golden era bodybuild favo...
...,...,...,...,...,...,...,...
744,Jump to pull-up,The jump to pull-up is a bodyweight exercise t...,Strength,Lats,Body Only,Intermediate,jump bodyweight exercis target muscl back bice...
745,Bent-over scapular slide,The bent-over scapular slide is an upper-body ...,Strength,Lats,Body Only,Intermediate,scapular slide exercis help scapular shoulder ...
746,Iron cross stretch,The iron cross stretch is a bodyweight stretch...,Strength,Lats,Body Only,Intermediate,iron cross stretch bodyweight stretch focus hi...
747,King Maker Pull-Up,The pull-up is a multi-joint bodyweight exerci...,Strength,Lats,Body Only,Intermediate,bodyweight exercis build strength muscl upper ...


In [223]:
pred("Assisted Chin-Up")

Band-assisted chin-up
Band-assisted pull-up
Chin-Up
TBS Chin-Up
Neutral-grip pull-up


In [224]:
pred("Chin-Up")

TBS Chin-Up
Neutral-grip pull-up
Assisted Chin-Up
Band-assisted chin-up
Band-assisted pull-up


In [225]:
pred("Neutral-grip pull-up")

Chin-Up
TBS Chin-Up
Band-assisted pull-up
Assisted Chin-Up
Band-assisted chin-up


In [226]:
pred("Crunch")

Crunches
Crunch - Gethin Variation
Bent-knee reverse crunch
Cross-Body Crunch
Elbow-to-knee crunch


In [227]:
pred("Neutral-grip pull-up")

Chin-Up
TBS Chin-Up
Band-assisted pull-up
Assisted Chin-Up
Band-assisted chin-up


In [228]:
import re

results = Cleaned_df[
    Cleaned_df["Title"].str.contains(
        r"rep.*push|push.*rep",
        case=False,
        na=False
    )
]

print(results["Title"])

500    1.5-rep push-up
Name: Title, dtype: object


In [229]:
num_results = Cleaned_df[
    Cleaned_df["Title"].str.contains(
        r"\d",
        na=False
    )
]

print(num_results["Title"])

18      Kettlebell 3-point leg extension
100                           3/4 sit-up
500                      1.5-rep push-up
949                    3D dumbbell lunge
1021                90-degree jump squat
Name: Title, dtype: object


In [230]:
pred("1.5-rep push-up")

Pushups
FYR Push-Up
Push-Up - Gethin Variation
Push-up
UP Push-up


In [235]:
with open("workout.pkl","wb") as f:
    pickle.dump(Cleaned_df,f)

In [236]:
with open("similarity.pkl","wb") as f:
    pickle.dump(similarity,f)

In [237]:
with open("vectorizer.pkl","wb") as f:
    pickle.dump(Vectorizer,f)

In [238]:
with open("vectors.pkl","wb") as f:
    pickle.dump(vectors,f)